In [1]:
import torch
import numpy as np
import os
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
import os
import numpy as np
from torch.utils.data import Dataset
from PIL import Image


class LensingDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = [os.path.join(root_dir, fname) for fname in os.listdir(root_dir) if fname.endswith(".npy")]

        if len(self.image_paths) == 0:
            raise RuntimeError(f"Error: No .npy files found in {root_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]

        # Load image from .npy file
        image = np.load(image_path)

        # Ensure correct dtype (convert to uint8 if necessary)
        if image.dtype != np.uint8:
            image = (255 * (image - image.min()) / (image.max() - image.min())).astype(np.uint8)

        # Ensure correct shape
        if len(image.shape) == 2:  # If grayscale (H, W)
            image = np.stack([image] * 3, axis=-1)  # Convert to (H, W, 3)

        elif len(image.shape) == 3 and image.shape[0] == 1:  # If (1, H, W)
            image = np.squeeze(image, axis=0)  # Convert (1, H, W) → (H, W)
            image = np.stack([image] * 3, axis=-1)  # Convert to (H, W, 3)

        elif len(image.shape) == 3 and image.shape[-1] == 1:  # If (H, W, 1)
            image = np.repeat(image, 3, axis=-1)  # Convert to (H, W, 3)

        # Convert to PIL Image
        image = Image.fromarray(image)

        # Apply transformation
        if self.transform:
            image = self.transform(image)

        return image

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from timm.models.vision_transformer import VisionTransformer
from torchvision.models.vision_transformer import VisionTransformer
import timm
import torch.nn as nn

class MaskedAutoencoder(nn.Module):
    def __init__(self):
        super(MaskedAutoencoder, self).__init__()
        self.encoder = timm.create_model(
            "vit_base_patch16_224",  # Predefined ViT model
            pretrained=False,        # We train from scratch
            num_classes=0            # No classification head
        )

        self.decoder = nn.Sequential(
            nn.Linear(768, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2048)
        )

    def forward(self, x, mask=None):  # Add mask as an optional argument
        if mask is not None:
            x = x * mask  # Apply mask to the input image

        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from mae_model import MaskedAutoencoder
from utilss import LensingDataset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.models as models
from sklearn.decomposition import PCA

# Function to create a binary mask
def create_mask(image_shape, mask_ratio=0.75):
    N, C, H, W = image_shape  # Get batch size and image dimensions
    mask = torch.rand(N, C, H, W) > mask_ratio  # Generate random mask
    return mask.float()

# Data transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load dataset
dataset = LensingDataset(root_dir="Dataset/no_sub", transform=transform)
print(f"Number of images loaded: {len(dataset)}")
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MaskedAutoencoder().to(device)

# Load a pretrained ResNet50 for feature extraction
resnet = models.resnet50(pretrained=True)
resnet.fc = torch.nn.Identity()  # Remove the final classification layer
resnet = resnet.to(device)
resnet.eval()

# Loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Number of images loaded: 29449


c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    total_loss = 0
    for imgs in dataloader:
        imgs = imgs.to(device)  # Move to GPU if available
        masks = create_mask(imgs.shape)  # Generate masks dynamically

        # Forward pass through MAE
        reconstructed_imgs = model(imgs)

        # Extract image features using ResNet
        with torch.no_grad():
            imgs_features = resnet(imgs)  # Expected shape: [batch_size, 2048]

        # Ensure reconstructed output matches feature dimensions
        projection_layer = nn.Linear(2048, 768).to(device)
        imgs_features_768 = projection_layer(imgs_features)

        # Compute loss
        loss = F.cosine_embedding_loss(reconstructed_imgs, imgs_features, torch.ones(imgs.shape[0]).to(imgs.device))

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(dataloader)}")